In [1]:
import os
import numpy as np
import tensorflow as tf
from utils import load_tiff_image
from sklearn.model_selection import train_test_split


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:

DATASET_PATH = "../dataset"
IMG_SIZE = 128

In [3]:
images = []
labels = []

In [5]:
for label, folder in enumerate(["NonMangroves", "Mangroves"]):
    folder_path = os.path.join(DATASET_PATH, folder)

    for file in os.listdir(folder_path):
        if file.endswith(".tif") or file.endswith(".tiff"):
            img_path = os.path.join(folder_path, file)
            img = load_tiff_image(img_path, (IMG_SIZE, IMG_SIZE))
            images.append(img)
            labels.append(label)

X = np.array(images) / 255.0
y = np.array(labels)


In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

In [8]:
base_model.trainable = False

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [9]:
model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [10]:
model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=16
)

Epoch 1/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 4s 61ms/step - accuracy: 0.8028 - loss: 0.4314 - val_accuracy: 0.8712 - val_loss: 0.3399
Epoch 2/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 0.8536 - loss: 0.3532 - val_accuracy: 0.8712 - val_loss: 0.3353
Epoch 3/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step - accuracy: 0.8536 - loss: 0.3542 - val_accuracy: 0.8712 - val_loss: 0.3438
Epoch 4/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step - accuracy: 0.8536 - loss: 0.3499 - val_accuracy: 0.8712 - val_loss: 0.3331
Epoch 5/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step - accuracy: 0.8536 - loss: 0.3502 - val_accuracy: 0.8712 - val_loss: 0.3433


In [11]:

model.save("mangrove_classifier.h5")